# Tool-Router Ladder: Stage E1, teacher-forced generation for the 25% / 50% adapters (E1-3)

The data-scaling curve's two new points: `tool-router-qwen15b-sft-f25` and `-f50` at catalogs 16, 80
and 40, on the same 1,416 steps as every forced-eval row. Same engine as the 100% point (the v1
1.5B rows: vLLM 0.30.0 on a T4), so the curve compares like with like.

**Settings:** GPU T4, Internet on, the tool-router dataset attached (the one with the prompts files),
secret `HF_TOKEN` ticked.
**Run:** Save Version → **Save & Run All (Commit)**. **No API quota.** About 4–5 GPU h for six runs
(1.5B is about half of 3B's 7.1 h). Each run is pushed to the Hub when it finishes, and a rerun
resumes from there, so a timed-out session loses nothing.

**Then on the Mac:** `make pull` and `make score-forced ADAPTER=qwen15b-sft-f25` (and `-f50`).

In [ ]:
# Secrets: Add-ons -> Secrets, and TICK each one for this notebook.
import os
from kaggle_secrets import UserSecretsClient
_s = UserSecretsClient()
for key in ["HF_TOKEN"]:
    os.environ[key] = _s.get_secret(key)
print("secrets loaded:", ["HF_TOKEN"])

In [ ]:
# Pinned: every forced-eval row so far is vLLM 0.30.0 (with a CUDA 13 torch).
# Kaggle's image now ships a CUDA 12.8 torchaudio, and transformers imports torchaudio
# whenever it is installed, so it crashes against that torch. Nothing here uses audio.
!pip install -q vllm==0.30.0
!pip uninstall -y -q torchaudio

# Step out of the repo BEFORE deleting it: rmtree on the kernel's own working
# directory leaves cwd pointing at a dead inode, and then every ! command fails
# with "getcwd: cannot access parent directories" -- including this git clone.
%cd /kaggle/working
import shutil
shutil.rmtree("/kaggle/working/the_llm_project", ignore_errors=True)
!git clone -q https://github.com/madhusiddharths/the_llm_project.git /kaggle/working/the_llm_project
%cd /kaggle/working/the_llm_project
!git log -1 --oneline

In [ ]:
# Finds the attached dataset wherever Kaggle mounted it (it must contain manifest.json).
import glob, os
hits = glob.glob("/kaggle/input/**/manifest.json", recursive=True)
assert hits, "attach the tool-router dataset (Add Input) - no manifest.json under /kaggle/input"
DATA = os.path.dirname(hits[0])
HF_USER = "madhusiddharths1"   # your Hugging Face username
print("DATA =", DATA); print(sorted(os.listdir(DATA)))

OUT = '/kaggle/working/out'
os.makedirs(OUT, exist_ok=True)

In [ ]:
# Resume across sessions: bring back any E1 completions an earlier session pushed.
import shutil
from huggingface_hub import HfApi, snapshot_download
REPO = f"{HF_USER}/tool-router-results"
api = HfApi()
api.create_repo(REPO, repo_type="dataset", private=True, exist_ok=True)
snap = snapshot_download(REPO, repo_type="dataset", allow_patterns=["completions/completions-qwen15b-sft-f*"])
for f in glob.glob(f"{snap}/completions/completions-qwen15b-sft-f*"):
    shutil.copy(f, OUT)
print("resuming from:", sorted(os.listdir(OUT)))

def push(path):
    for f in (path, path.replace(".jsonl", ".meta.json")):
        if os.path.exists(f):
            api.upload_file(path_or_fileobj=f, path_in_repo=f"completions/{os.path.basename(f)}",
                            repo_id=REPO, repo_type="dataset")

In [ ]:
# One run at a time, pushed as soon as it finishes, so a session that times out keeps its work.
runs = [(tag, c) for tag in ("f25", "f50") for c in (16, 80, 40)]
for tag, catalog in runs:
    out = f"{OUT}/completions-qwen15b-sft-{tag}-c{catalog}.jsonl"
    prompts = f"{DATA}/prompts-c{catalog}.jsonl"
    print(f"===== qwen15b-sft-{tag} c{catalog} =====")
    !python src/generate.py --config configs/qwen15b.yaml --adapter "$HF_USER/tool-router-qwen15b-sft-{tag}" --prompts "{prompts}" --out "{out}"
    # `!` does not raise on failure: stop here instead of running (and pushing) the rest.
    assert _exit_code == 0, f"generate.py failed on qwen15b-sft-{tag} c{catalog}: send the output above"
    push(out)
print("pushed:", sorted(os.listdir(OUT)))